# CT EDA — Mayo Clinic Sparse-View CT
Exploratory data analysis for the CT dataset.
Run this **before training** to verify data paths and inspect the raw images.

**Prerequisites:** CT data accessible via `CT_DATA_DIR` in `config.py`.

In [ ]:
# ── Box token (only needed until slices are cached) ─────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath("../..."))
from config import BOX_TOKEN

# Paste your token here OR set BOX_TOKEN in config.py
# Regenerate every 60 min at https://developer.box.com if slices are not yet cached.
MY_BOX_TOKEN = BOX_TOKEN or "PASTE_YOUR_BOX_DEV_TOKEN_HERE"

# This will fetch ONLY the configured slices (30 train + 10 val + 10 test per patient)
# via HTTP range requests — NO full zip download. Cached to ct_cache/slices/.
from ct.dataset import BoxCTDataset
from config import CT

# Quick check: fetch one patient's train slices to confirm the token works
ds = BoxCTDataset(
    box_token=MY_BOX_TOKEN,
    patients=[CT["train_patients"][0]],
    slices_per_patient=2,  # just 2 slices to verify
)
print("Box streaming OK — token is valid and slices are accessible.")


In [ ]:
# ── Download CT data from Box (run once; skips if already present) ──────────
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
from ct.dataset import download_ct_data

BOX_DEV_TOKEN = """PASTE_YOUR_BOX_DEV_TOKEN_HERE"""
# Token expires every 60 min — regenerate at https://developer.box.com

# This is identical to the download cell in the original notebook.
# After this cell, the data is cached locally and never needs to be re-downloaded.
# extract_dir = download_ct_data(BOX_DEV_TOKEN)
# print("Data ready at:", extract_dir)

# Uncomment the two lines above to download.  Skip if data already exists.


## 1. Patient / Slice Inventory

In [ ]:
all_patients = sorted(CT_DATA_DIR.iterdir()) if CT_DATA_DIR.exists() else []
inventory    = {}
total        = 0

print(f"{'Patient':10s} {'Split':6s} {'Slices':>7s}")
print("-" * 28)
for p in all_patients:
    slices = sorted((p / "full_3mm").glob("*.IMA")) if (p / "full_3mm").exists() else []
    inventory[p.name] = slices
    total += len(slices)
    split = "TRAIN" if p.name in CT["train_patients"] else             "VAL"   if p.name in CT["val_patients"]   else             "TEST"  if p.name in CT["test_patients"]  else "?"
    print(f"{p.name:10s} {split:6s} {len(slices):7d}")
print(f"{'Total':10s} {'':6s} {total:7d}")


## 2. Sample Slices — Ground Truth

In [ ]:
import sys
sys.path.insert(0, os.path.abspath('..'))
from dataset import load_ima_file

patients_to_show = [p for p in CT["train_patients"] if p in inventory][:4]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col, pid in enumerate(patients_to_show):
    slices = inventory[pid]
    mid    = len(slices) // 2
    for row, idx in enumerate([mid, min(mid + 10, len(slices) - 1)]):
        img = load_ima_file(slices[idx])
        axes[row, col].imshow(img, cmap="gray")
        axes[row, col].set_title(f"{pid} slice {idx}", fontsize=9)
        axes[row, col].axis("off")

plt.suptitle("Sample CT Slices  (normalised [0,1])", fontsize=14)
plt.tight_layout()
plt.savefig("ct_eda_samples.png", dpi=120, bbox_inches="tight")
plt.show()


## 3. Effect of Number of Views on FBP Quality

In [ ]:
from dataset import make_sparse_sinogram

pid    = patients_to_show[0]
slices = inventory[pid]
gt     = load_ima_file(slices[len(slices) // 2])

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, n_views in zip(axes, [10, 30, 60, 180]):
    _, fbp = make_sparse_sinogram(gt, n_views)
    ax.imshow(fbp, cmap="gray")
    ax.set_title(f"{n_views} views — FBP", fontsize=12)
    ax.axis("off")

plt.suptitle("FBP Quality vs Number of Projection Views", fontsize=14)
plt.tight_layout()
plt.savefig("ct_eda_fbp_views.png", dpi=120, bbox_inches="tight")
plt.show()


## 4. Sinogram Visualisation (60 views)

In [ ]:
sino_60, fbp_60 = make_sparse_sinogram(gt, 60)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(gt,      cmap="gray");              axes[0].set_title("Ground Truth");       axes[0].axis("off")
axes[1].imshow(sino_60, cmap="gray", aspect="auto"); axes[1].set_title("Sinogram (60 views)"); axes[1].axis("off")
axes[2].imshow(fbp_60,  cmap="gray");              axes[2].set_title("FBP (60 views)");     axes[2].axis("off")

plt.suptitle("Sinogram and FBP Reconstruction (60-view Sparse CT)", fontsize=14)
plt.tight_layout()
plt.savefig("ct_eda_sinogram.png", dpi=120, bbox_inches="tight")
plt.show()


## 5. Pixel Intensity Histograms by Split

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (split_name, patients) in zip(
    axes,
    [("Train", CT["train_patients"]),
     ("Val",   CT["val_patients"]),
     ("Test",  CT["test_patients"])]
):
    sample_pix = []
    for pid in patients[:2]:
        for sl in inventory.get(pid, [])[:5]:
            sample_pix.extend(load_ima_file(sl).flatten().tolist())
    ax.hist(sample_pix, bins=100, color="steelblue", alpha=0.85)
    ax.set_title(f"{split_name} split", fontsize=12)
    ax.set_xlabel("Normalised HU [0, 1]")
    ax.set_ylabel("Count")
    ax.grid(alpha=0.3)

plt.suptitle("CT Pixel Intensity Distributions", fontsize=14)
plt.tight_layout()
plt.savefig("ct_eda_histograms.png", dpi=120, bbox_inches="tight")
plt.show()


## 6. DataLoader Sanity Check

In [ ]:
from dataset import build_ct_loaders

train_loader, val_loader, test_loader = build_ct_loaders()
fbp_b, sino_b, gt_b = next(iter(train_loader))

print(f"FBP   shape : {fbp_b.shape}  range [{fbp_b.min():.3f}, {fbp_b.max():.3f}]")
print(f"Sino  shape : {sino_b.shape}")
print(f"GT    shape : {gt_b.shape}  range [{gt_b.min():.3f}, {gt_b.max():.3f}]")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(axes,
    [fbp_b[0, 0], sino_b[0, 0], gt_b[0, 0]],
    ["FBP (input)", "Sinogram", "Ground Truth"]):
    ax.imshow(img.numpy(), cmap="gray"); ax.set_title(title); ax.axis("off")
plt.suptitle("DataLoader batch sample", fontsize=13)
plt.tight_layout()
plt.savefig("ct_eda_dataloader.png", dpi=120, bbox_inches="tight")
plt.show()
print("DataLoader OK")


## 7. FBP Baseline Metrics on Test Set

In [ ]:
import sys
sys.path.insert(0, os.path.abspath('../..'))
from shared.metrics import compute_metrics
import torch, numpy as np

fbp_psnrs, fbp_ssims, fbp_rmses = [], [], []
with torch.no_grad():
    for fbp, sino, gt in test_loader:
        for p, g in zip(fbp.squeeze(1).numpy(), gt.squeeze(1).numpy()):
            m = compute_metrics(p, g)
            fbp_psnrs.append(m["PSNR"])
            fbp_ssims.append(m["SSIM"])
            fbp_rmses.append(m["RMSE"])

print(f"FBP baseline ({len(fbp_psnrs)} test slices):")
print(f"  PSNR : {np.mean(fbp_psnrs):.3f} +/- {np.std(fbp_psnrs):.3f} dB")
print(f"  SSIM : {np.mean(fbp_ssims):.4f} +/- {np.std(fbp_ssims):.4f}")
print(f"  RMSE : {np.mean(fbp_rmses):.5f} +/- {np.std(fbp_rmses):.5f}")
